# TP4: Algorithmes gloutons

## Introduction et solutions possibles

**Question 1.** En suivant le critère de gloutonnerie « objets de plus grande valeur » d'abord, on prendrait le collier et le candelabre. Le poids total serait de 6 et la valeur de 25. Ce choix n'est pas optimal : prendre le collier, le Cormen et le glomorphe donne un poids de 8 (maximal) et une valeur de 29 > 25.

**Question 2.** Il faut commencer par déterminer le nombre de combinaisons possibles. Il est de $2^4=16$ possibilités car pour chaque objet, on a deux états possibles : choisi ou non. Bien entendu, un grand nombre de ces combinaisons ne sont pas admissibles car leur poids dépasserait 8kg. Si on considère qu'il faut passer 5 secondes par combinaison, alors il faudra $5*16=80$ secondes pour se décider.

## Solution gloutonne en Python

In [1]:
def rapport_valeur_poids(valeurs, poids):
    assert(len(valeurs) == len(poids))
    return [valeurs[i]/poids[i] for i in range(len(valeurs))]
    # Equivalent à:
    # alphas = []
    # for i in range(len(valeurs)):
    #     alphas.append(valeurs[i]/poids[i])
    # return alpha

In [2]:
def meilleur_objet_restant(alpha, traites):
    assert(len(alpha) == len(traites))
    maximum = 0
    idx_maximum = -1
    
    for idx in range(len(alpha)):
        if not traites[idx] and alpha[idx] > maximum:
            maximum = alpha[idx]
            idx_maximum = idx
    
    return idx_maximum

In [3]:
def glouton(valeurs, poids, C):
    C_restant = C
    alpha = rapport_valeur_poids(valeurs, poids)
    n = len(valeurs)
    traites = [False for _ in range(n)]

    resultat = []
    valeur_totale = 0
    
    idx_while = 0
    while C_restant > 0 and idx_while < n:
        idx_meilleur_objet = meilleur_objet_restant(alpha, traites)

        if idx_meilleur_objet != -1:
            traites[idx_meilleur_objet] = True
            
            if poids[idx_meilleur_objet] <= C_restant:
                C_restant -= poids[idx_meilleur_objet]
                resultat.append(idx_meilleur_objet)
                valeur_totale += valeurs[idx_meilleur_objet]
        
        idx_while += 1

    return resultat, valeur_totale

In [4]:
def afficher_sortie_glouton(resultat, valeur_totale, noms_objets):
    a_afficher = "Les objets volés sont : "
    
    for idx_resultat in resultat:
        a_afficher += noms_objets[idx_resultat] + ", "

    a_afficher = a_afficher[:-2] + ". La valeur totale est : " + str(valeur_totale*100) + " €."
    print(a_afficher)

## Analyse de la solution gloutonne

In [5]:
noms_objets = ["collier", "candelable", "Cormen", "glomorphe"]
poids = [1, 5, 3, 4]
valeurs = [15, 10, 9, 5]

resultat, valeur_totale = glouton(valeurs, poids, C=8)
afficher_sortie_glouton(resultat, valeur_totale, noms_objets)

Les objets volés sont : collier, Cormen, glomorphe. La valeur totale est : 2900 €.


La solution est en effet optimale cette fois-ci. Dans cette configuration, le critère de gloutonnerie choisi est donc meilleur.

In [6]:
noms_objets = ["objet_lourd", "objet_leger_1", "objet_leger_2"]
poids = [5, 4, 4]
valeurs = [10, 7, 7]

resultat, valeur_totale = glouton(valeurs, poids, C=8)
afficher_sortie_glouton(resultat, valeur_totale, noms_objets)

Les objets volés sont : objet_lourd. La valeur totale est : 1000 €.


Dans cette situation précise, un seul objet de poids le plus important (5) est choisi, empêchant la sélection d'autres objets (peu importe le second objet que l'on choisirait, le poids total deviendrait $5+4=9>8$). La valeur totale de cette solution est alors de $1000$€, or en choisisant objet_leger_1 et objet_leger_2, on aurait un poids total de $8$ et une valeur totale de $1400$€$ > 1000$€.

**Question 9.** Posons $n$ le nombre d'objets. Dans la fonction « glouton », tout ce qui est avant le while est au maximum en $O(n)$. Dans le pire des cas, on itère $n$ fois dans la boucle while. Dans cette boucle :
- La première ligne est l'appel à meilleur_objet_restant. En regardant cette fonction, on déduit rapidement que sa complexité est en $O(n)$, avec une boucle itérée $n$ fois contenant des lignes à opérations en temps constant, et en dehors de la boucle, des opérations en temps constant.
- On exécute des opérations en temps constant $O(1)$ du premier if à la ligne « idx_while += 1 ».

En faisant le bilan, le nombre total d'opérations élémentaires $f(n) \in O(n) + n \times O(n) = O\left(n + n^2\right) = O\left(n^2\right)$.

In [7]:
def glouton_rapide(valeurs, poids, C):
    C_restant = C
    
    alphas_et_indices = [(valeurs[i]/poids[i], i) for i in range(len(valeurs))] # On a besoin de se souvenir à quel objet correspond chaque rapport v_i/p_i. La fonction sorted trie par défaut les éléments d'une liste de n-uplets en fonction de la première valeur de ce n-uplet : on aura bien l'ordre désiré.
    alphas_et_indices = sorted(alphas_et_indices, reverse = True)
    n = len(valeurs)

    indices_objets_choisis = []
    valeur_totale = 0

    for alpha, i in alphas_et_indices: # On pourrait aussi écrire for alpha_et_indice in alphas_et_indices, puis dans le for, faire alpha, i = alpha_et_indice; voire même l'écrire en deux lignes
        if poids[i] <= C_restant:
            C_restant -= poids[i]
            indices_objets_choisis.append(i)
            valeur_totale += valeurs[i]
    
    return indices_objets_choisis, valeur_totale

In [8]:
noms_objets = ["collier", "candelable", "Cormen", "glomorphe"]
poids = [1, 5, 3, 4]
valeurs = [15, 10, 9, 5]

resultat, valeur_totale = glouton_rapide(valeurs, poids, C=8)
afficher_sortie_glouton(resultat, valeur_totale, noms_objets)

Les objets volés sont : collier, Cormen, glomorphe. La valeur totale est : 2900 €.


Vérifions que la solution proposée est bien en $O\left(n\log n\right)$. Toutes les lignes jusque la boucle pour sont en temps constant $O(1)$ excepté la ligne incluant « sorted », qui elle, comme indiqué dans l'énoncé, est en $O\left(n\log n\right)$. La boucle for itère de $(n-1)$ à $0$ inclus, soit $n$ itérations. Au sein de cette boucle, toutes les opérations se font en temps constant $O(1)$. Enfin, après la boucle pour, les retours se font en $O(1)$.

En faisant le bilan : $f(n) \in O(1) + O\left(n\log n\right) + n \times O(1) = O\left(n + n\log n\right) = O\left(n\log n\right)$. 

Cette solution est en effet plus efficace temporellement parlant.